# The Electric Field Instrument (EFI) for THEMIS — Implementation / 구현

**Paper**: Bonnell, J.W., Mozer, F.S., Delory, G.T., Hull, A.J., Ergun, R.E., Cully, C.M., Angelopoulos, V., Harvey, P.R., *The Electric Field Instrument (EFI) for THEMIS*, Space Sci. Rev. **141**, 303–341, 2008. DOI: 10.1007/s11214-008-9469-2

This notebook reproduces four core algorithms from the THEMIS–EFI instrument paper using synthetic data:
1. **Double-probe E-field formula and 20 m wire-boom geometry** — $E = -(V_n - V_m)/L$ with the 49.6 m / 40.4 m / 6.93 m baselines, including spin-modulated despin.
2. **Spacecraft sheath / floating-potential model** — photoelectron + ambient electron + ambient ion currents on a 8-cm graphite sphere; bias-current optimisation to minimise sheath resistance $R_s$.
3. **$\mathbf{E}\times\mathbf{B}$ drift from FGM + EFI cross-calibration** — compare $\mathbf{E}_{EFI}$ against $-\mathbf{V}_i\times\mathbf{B}$ to recover the boom-shorting factor and confirm ideal-MHD agreement.
4. **Substorm-onset E-field signature** — the inflow-region transverse E that grows by ~1 mV/m at plasma-sheet onset (8–10 R$_E$).

이 노트북은 THEMIS–EFI 논문의 핵심 알고리즘 네 가지를 합성 데이터로 재현한다: (1) double-probe E-field 식과 20 m wire-boom 기하, (2) 우주선 sheath / floating-potential 모델, (3) FGM+EFI 결합으로부터 $\mathbf{E}\times\mathbf{B}$ 드리프트와 boom-shorting 교정, (4) plasma sheet inflow에서의 substorm-onset E-field 시그니쳐.

**Conda env**: `study-with-ai`. **Libraries**: NumPy, SciPy, Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from typing import Tuple
from scipy import constants
from scipy.optimize import brentq

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
rng = np.random.default_rng(seed=42)

# Useful physical constants (SI)
K_B = constants.k          # Boltzmann
Q_E = constants.e          # elementary charge
M_E = constants.m_e        # electron mass
M_P = constants.m_p        # proton mass
EPS0 = constants.epsilon_0
EV = constants.e           # 1 eV in J

# THEMIS-EFI constants from Bonnell et al. (2008) Table 1
L_SPB_LONG = 49.6          # m, X-axis (V1-V2) tip-to-tip
L_SPB_SHORT = 40.4         # m, Y-axis (V3-V4) tip-to-tip
L_AXB = 6.93               # m, Z-axis (V5-V6) tip-to-tip
R_SPHERE = 0.08 / 2.0      # m, 8-cm-diameter sphere => 4-cm radius
A_SPHERE = 4.0 * np.pi * R_SPHERE**2  # m^2, total surface area
A_SPHERE_CROSS = np.pi * R_SPHERE**2  # m^2, illuminated cross-section

## Part 1: Double-probe E-field formula and 20 m wire-boom geometry / 더블 프로브 식과 20 m wire-boom 기하

For two sphere sensors at positions $\mathbf{X}_n$ and $\mathbf{X}_m$ in a uniform external field $\mathbf{E}$, the differential potential is

$$V_n - V_m = -\mathbf{E}\cdot(\mathbf{X}_n - \mathbf{X}_m).$$

With baseline length $L = |\mathbf{X}_n - \mathbf{X}_m|$ and unit baseline vector $\hat{\mathbf{l}}$, the E-field component along that baseline is $E_\parallel = -(V_n - V_m)/L$. THEMIS deploys three orthogonal pairs in the spacecraft spin frame: $L_{12}=49.6$ m (X), $L_{34}=40.4$ m (Y), $L_{56}=6.93$ m (Z, axial).

On orbit, the *effective* baseline is shortened by the **boom-shorting factor** $f_{sh} \in [1.3, 1.6]$, because the partially conducting spacecraft body and grounded cable braid distort the equipotential surfaces between the spheres. The corrected E is

$$E_{true} = f_{sh}\,E_{measured}.$$

두 sphere sensor의 차동 전위 $V_n-V_m$는 외부 E-field와 삼면체 baseline의 내적으로 주어진다. THEMIS는 세 직교 쌍 (X=49.6 m, Y=40.4 m, Z=6.93 m)을 배치하며, 권역 boom shorting (1.3–1.6) 보정이 필요하다.

In [ ]:
def double_probe_E(
    V_n: NDArray,
    V_m: NDArray,
    baseline_m: float,
    shorting_factor: float = 1.0,
) -> NDArray:
    """Compute the E-field component along a double-probe baseline.

    Implements the Bonnell et al. (2008) Sec. 1 baseline formula
    ``E = -(V_n - V_m) / L`` with optional boom-shorting correction
    ``E_true = shorting_factor * E_measured``.

    Args:
        V_n: Floating potential of sensor n (V), scalar or array.
        V_m: Floating potential of sensor m (V), same shape as V_n.
        baseline_m: Tip-to-tip baseline length L between spheres (m).
        shorting_factor: Multiplicative correction for partial sensor
            shorting; THEMIS measured 1.3-1.6 (default 1.0 = no correction).

    Returns:
        Component of E along the baseline (V/m).
    """
    return -shorting_factor * (V_n - V_m) / baseline_m


def synthesise_spin_potentials(
    spin_phase_rad: NDArray,
    Ex_DSL: float,
    Ey_DSL: float,
    baseline_m: float,
    sunward_offset_mV: float = 2.5,
    noise_mV: float = 0.5,
    seed: int = 0,
) -> Tuple[NDArray, NDArray]:
    """Synthesise (V_n, V_m) for an X- or Y-pair under a uniform DSL field.

    The differential potential is the sinusoidal projection of the DSL
    horizontal field onto the rotating boom: ``Vn - Vm = -L*(Ex*cos(psi) + Ey*sin(psi))``.
    A spin-symmetric photoelectron asymmetry of ``sunward_offset_mV`` is added to
    Vn (the sunward sensor over half the spin) to reproduce the well-known
    THEMIS Vsun-side offset (Sec. 4.4).

    Args:
        spin_phase_rad: Sun-pulse-referenced spin phase psi (rad), 0 <= psi < 2*pi.
        Ex_DSL: True DSL X component of E in mV/m.
        Ey_DSL: True DSL Y component of E in mV/m.
        baseline_m: Tip-to-tip boom length (m).
        sunward_offset_mV: Per-spin sunward photoelectron asymmetry in mV.
        noise_mV: White Gaussian noise on each per-sensor reading.
        seed: RNG seed for reproducibility.

    Returns:
        (V_n, V_m) arrays in volts.
    """
    local_rng = np.random.default_rng(seed)
    Ex = Ex_DSL * 1e-3  # V/m
    Ey = Ey_DSL * 1e-3
    # Differential potential signature
    dV = -baseline_m * (Ex * np.cos(spin_phase_rad) + Ey * np.sin(spin_phase_rad))
    # Symmetric mean potential (s/c floating, ~0 here for clarity)
    V_mean = np.zeros_like(dV)
    V_n = V_mean + 0.5 * dV + sunward_offset_mV * 1e-3
    V_m = V_mean - 0.5 * dV
    V_n += noise_mV * 1e-3 * local_rng.standard_normal(dV.shape)
    V_m += noise_mV * 1e-3 * local_rng.standard_normal(dV.shape)
    return V_n, V_m


def spin_fit_E(
    spin_phase_rad: NDArray,
    E_along_boom: NDArray,
) -> Tuple[float, float, float]:
    """Fit ``E(psi) = A + B*sin(psi) + C*cos(psi)`` to despun E.

    Implements the THEMIS spin-fit model (Bonnell et al. 2008, Sec. 2.3):
    A is the per-spin offset, while B and C give the despun horizontal field.

    Args:
        spin_phase_rad: Sun-pulse-referenced spin phase psi (rad).
        E_along_boom: Measured E along the rotating boom (mV/m).

    Returns:
        (A, B, C) coefficients of the spin-fit model in mV/m.
    """
    M = np.column_stack([
        np.ones_like(spin_phase_rad),
        np.sin(spin_phase_rad),
        np.cos(spin_phase_rad),
    ])
    coeffs, *_ = np.linalg.lstsq(M, E_along_boom, rcond=None)
    return float(coeffs[0]), float(coeffs[1]), float(coeffs[2])


# --- Demo: 3 s spacecraft spin period, 100 ms cadence => 30 spins of phase
spin_period_s = 3.0
fs_hz = 128.0       # standard EDC sample rate
T_total_s = 60.0
t = np.arange(0, T_total_s, 1.0 / fs_hz)
psi = 2.0 * np.pi * (t / spin_period_s) % (2.0 * np.pi)

Ex_true_mVm = 1.5
Ey_true_mVm = -0.8

V1, V2 = synthesise_spin_potentials(psi, Ex_true_mVm, Ey_true_mVm, L_SPB_LONG, seed=1)
V3, V4 = synthesise_spin_potentials(psi, Ex_true_mVm, Ey_true_mVm, L_SPB_SHORT, seed=2)

E12 = double_probe_E(V1, V2, L_SPB_LONG) * 1e3   # mV/m
E34 = double_probe_E(V3, V4, L_SPB_SHORT) * 1e3  # mV/m

A12, B12, C12 = spin_fit_E(psi, E12)
A34, B34, C34 = spin_fit_E(psi, E34)

print(f"True (Ex, Ey) DSL              : ({Ex_true_mVm:+.3f}, {Ey_true_mVm:+.3f}) mV/m")
print(f"Spin-fit recovery from E12 long: A={A12:+.3f}, B={B12:+.3f}, C={C12:+.3f} mV/m")
print(f"Spin-fit recovery from E34 short: A={A34:+.3f}, B={B34:+.3f}, C={C34:+.3f} mV/m")
print(f"  (model: E_along = -Ex*cos(psi) - Ey*sin(psi)  =>  B == -Ey, C == -Ex)")

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(t, E12, lw=0.7, label="$E_{12}$ (49.6 m)")
ax[0].plot(t, E34, lw=0.7, label="$E_{34}$ (40.4 m)", alpha=0.8)
ax[0].set_ylabel("E along boom [mV/m]")
ax[0].set_title("Spin-modulated double-probe E-field signature (synthetic)")
ax[0].legend()
ax[0].grid(alpha=0.3)

fit_E12 = A12 + B12 * np.sin(psi) + C12 * np.cos(psi)
ax[1].plot(t[:300], E12[:300], lw=0.6, label="data")
ax[1].plot(t[:300], fit_E12[:300], lw=2, label="spin fit")
ax[1].set_xlabel("Time [s]")
ax[1].set_ylabel("$E_{12}$ [mV/m]")
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 1.b Boom-shorting recovery from long–short asymmetry / Long–short 비대칭으로부터 boom-shorting 회수

Because the long pair (49.6 m) has a smaller fractional boom-shorting effect than the short pair (40.4 m), comparing the spin-fit amplitudes lets us infer the underlying shorting factor without an *a priori* model. In the simple linear model both pairs measure the same $\mathbf{E}_{true}$, so equal $\sqrt{B^2+C^2}/L$ in *true* units means **identical recoveries when both are corrected by the same factor**. We illustrate the diagnostic by injecting a 1.45 shorting factor and recovering it.

Long pair의 boom-shorting 영향이 short pair보다 작기 때문에, 두 spin-fit 진폭을 비교하면 shorting factor를 직접 추정할 수 있다.

In [ ]:
def apply_boom_shorting(
    E_along_boom_mVm: NDArray,
    shorting_factor: float,
) -> NDArray:
    """Reduce the *measured* E-field amplitude by 1/shorting_factor.

    Models the on-orbit fact that THEMIS-EFI sees ``E_meas = E_true / f_sh``
    because partial conduction through the spacecraft body shortens the
    effective probe separation (Sec. 4.4 of Bonnell et al. 2008).

    Args:
        E_along_boom_mVm: True E along boom in mV/m.
        shorting_factor: Boom-shorting factor f_sh; > 1 reduces amplitude.

    Returns:
        Measured (reduced) E in mV/m.
    """
    return E_along_boom_mVm / shorting_factor


f_sh_inject = 1.45
E12_meas = apply_boom_shorting(E12, f_sh_inject)
E34_meas = apply_boom_shorting(E34, f_sh_inject)

_, B12m, C12m = spin_fit_E(psi, E12_meas)
_, B34m, C34m = spin_fit_E(psi, E34_meas)

amp12_meas = np.hypot(B12m, C12m)
amp34_meas = np.hypot(B34m, C34m)
amp_true = np.hypot(Ex_true_mVm, Ey_true_mVm)

f_sh_recovered_long = amp_true / amp12_meas
f_sh_recovered_short = amp_true / amp34_meas
print(f"Injected shorting factor      : {f_sh_inject:.3f}")
print(f"Recovered (long pair, 49.6 m) : {f_sh_recovered_long:.3f}")
print(f"Recovered (short pair, 40.4 m): {f_sh_recovered_short:.3f}")
print(f"|true E|        : {amp_true:.3f} mV/m")
print(f"|measured E_12| : {amp12_meas:.3f} mV/m  (long boom)")
print(f"|measured E_34| : {amp34_meas:.3f} mV/m  (short boom)")

## Part 2: Spacecraft sheath / floating-potential model / 우주선 sheath / floating-potential 모델

An isolated conductor in plasma reaches a floating potential $V_f$ where the *net* current $I(V) = I_e(V) + I_i(V) + I_{ph}(V)$ vanishes. For a sunlit graphite-coated 8-cm sphere in tenuous magnetospheric plasma:

- **Ambient electron current** (attracted when $V>0$):
$$I_e(V) = -e n_e A v_{th,e}\;f_e(V),\quad v_{th,e}=\sqrt{k_B T_e/(2\pi m_e)},\quad f_e = e^{eV/k_B T_e}\;(V<0)\text{ or }1+eV/k_B T_e\;(V>0).$$
- **Ambient ion current** (repelled when $V>0$):
$$I_i(V) = +e n_i A v_{th,i}\;f_i(V).$$
- **Photoemission saturation current** $I_{ph,sat}\approx J_{ph}\,A_{cross}$ with $J_{ph}\approx 4$ nA/cm$^2$ for sunlit graphite. For $V>0$ photoelectrons are pulled back and $I_{ph}(V) \approx I_{ph,sat}\, e^{-V/V_{ph}}$ with $V_{ph}\sim 1.5$ V; for $V<0$ all are emitted: $I_{ph}=I_{ph,sat}$.

The instrument adds an active **bias current** $I_{BIAS}$ (negative = electron injection) to flatten the I–V curve at the operating point. The small-signal sheath resistance is $R_s = (dI/dV)^{-1}$. Sec. 4.1 of Bonnell et al. shows that $R_s$ is minimised when $I_{BIAS}\sim\tfrac{1}{2}I_{ph,sat}$ — we reproduce this.

우주선 sphere는 아이온 / 전자 / 광전자 세 전류의 합이 0이 되는 floating potential에서 평형을 이룬다. 적절한 너가티브 bias 전류를 주입하면 sheath 저항 $R_s$가 최대 100배 감소한다 (본 논문 Sec 4.1).

In [ ]:
def thermal_speed_avg(T_eV: float, mass_kg: float) -> float:
    """Mean thermal speed for one-sided flux through a plane (m/s).

    Returns ``v_th = sqrt(8 k T / (pi m)) / 4 = sqrt(k T / (2 pi m))``,
    the averaged speed used in OML probe-current expressions.

    Args:
        T_eV: Temperature in eV.
        mass_kg: Particle mass (kg).

    Returns:
        Thermal speed in m/s.
    """
    T_K = T_eV * EV / K_B
    return np.sqrt(K_B * T_K / (2.0 * np.pi * mass_kg))


def probe_current_components(
    V: NDArray,
    n_cm3: float,
    Te_eV: float,
    Ti_eV: float,
    A_m2: float,
    A_cross_m2: float,
    J_ph_nApcm2: float = 4.0,
    V_ph_V: float = 1.5,
) -> Tuple[NDArray, NDArray, NDArray]:
    """Compute electron, ion and photoemission currents on a spherical sensor.

    Uses the standard OML expressions for an attracted/repelled Maxwellian
    plasma plus a saturated photoemission with exponential return for V>0.
    Sign convention: positive current = positive charge flow onto the sphere.

    Args:
        V: Sphere potential w.r.t. plasma (V), array.
        n_cm3: Plasma density (cm^-3); assumes n_e = n_i.
        Te_eV: Electron temperature (eV).
        Ti_eV: Ion temperature (eV).
        A_m2: Total sphere surface area (m^2).
        A_cross_m2: Cross-sectional (sunlit) area (m^2).
        J_ph_nApcm2: Photoemission saturation current density (nA/cm^2).
        V_ph_V: Effective photoelectron temperature (V) for return current.

    Returns:
        (I_e, I_i, I_ph) currents in Amperes (signed).
    """
    n_m3 = n_cm3 * 1e6
    v_te = thermal_speed_avg(Te_eV, M_E)
    v_ti = thermal_speed_avg(Ti_eV, M_P)
    Te_J = Te_eV * EV
    Ti_J = Ti_eV * EV

    # Electrons: attracted when V>0 (linear), repelled when V<0 (Boltzmann).
    f_e = np.where(V >= 0, 1.0 + Q_E * V / Te_J, np.exp(Q_E * V / Te_J))
    I_e = -Q_E * n_m3 * A_m2 * v_te * f_e  # negative (electrons make negative current onto sphere)

    # Ions: repelled when V>0 (Boltzmann), attracted when V<0 (linear).
    f_i = np.where(V >= 0, np.exp(-Q_E * V / Ti_J), 1.0 - Q_E * V / Ti_J)
    I_i = +Q_E * n_m3 * A_m2 * v_ti * f_i

    # Photoemission: saturated for V<0, exponential return for V>0.
    J_ph_Apm2 = J_ph_nApcm2 * 1e-9 * 1e4  # nA/cm^2 -> A/m^2
    I_ph_sat = J_ph_Apm2 * (A_cross_m2 * 1.0)
    I_ph = np.where(V >= 0, I_ph_sat * np.exp(-V / V_ph_V), I_ph_sat * np.ones_like(V))

    return I_e, I_i, I_ph


def find_floating_potential(
    n_cm3: float,
    Te_eV: float,
    Ti_eV: float,
    A_m2: float,
    A_cross_m2: float,
    I_bias_A: float = 0.0,
    J_ph_nApcm2: float = 4.0,
) -> float:
    """Solve for V at which net current (incl. bias) is zero.

    Args:
        n_cm3: Plasma density (cm^-3).
        Te_eV: Electron temperature (eV).
        Ti_eV: Ion temperature (eV).
        A_m2: Sphere total area (m^2).
        A_cross_m2: Sphere cross-section (m^2).
        I_bias_A: Applied bias current (A); negative = injecting electrons
            (positive bias of operating point).
        J_ph_nApcm2: Photoemission saturation current density (nA/cm^2).

    Returns:
        Floating potential V_f (V).
    """
    def net_current(V):
        I_e, I_i, I_ph = probe_current_components(
            np.atleast_1d(V), n_cm3, Te_eV, Ti_eV, A_m2, A_cross_m2,
            J_ph_nApcm2=J_ph_nApcm2,
        )
        return float(I_e[0] + I_i[0] + I_ph[0] + I_bias_A)
    return brentq(net_current, -50.0, 50.0)


# CPS-like conditions from Bonnell et al. (2008) Sec. 4.1, Fig. 14:
# n=0.3 cm^-3, Te=600 eV, Ti=4.2 keV, sunlit graphite sphere
n_cps, Te_cps, Ti_cps = 0.3, 600.0, 4200.0

V_grid = np.linspace(-30, 30, 600)
I_e, I_i, I_ph = probe_current_components(
    V_grid, n_cps, Te_cps, Ti_cps, A_SPHERE, A_SPHERE_CROSS
)
I_total_unbiased = I_e + I_i + I_ph

V_f_unbiased = find_floating_potential(
    n_cps, Te_cps, Ti_cps, A_SPHERE, A_SPHERE_CROSS, I_bias_A=0.0
)
print(f"Photoemission saturation current : {(4.0e-9*1e4)*A_SPHERE_CROSS*1e9:.1f} nA")
print(f"Floating potential (no bias)     : {V_f_unbiased:+.2f} V (sunlit, CPS)")

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(V_grid, np.abs(I_e) * 1e9, label="|$I_e$|")
ax.semilogy(V_grid, np.abs(I_i) * 1e9, label="|$I_i$|")
ax.semilogy(V_grid, np.abs(I_ph) * 1e9, label="|$I_{ph}$|")
ax.semilogy(V_grid, np.abs(I_total_unbiased) * 1e9, "k--", label="|$I_{net}$|")
ax.axvline(V_f_unbiased, color="r", ls=":", label=f"$V_f$={V_f_unbiased:+.1f} V")
ax.set_xlabel("Sphere potential V [V]")
ax.set_ylabel("|Current| [nA]")
ax.set_title("Sunlit-sphere current components, CPS conditions (cf. Fig. 14)")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### 2.b Sheath resistance vs bias current / Bias 전류에 따른 sheath 저항

We sweep $I_{BIAS}$ over the THEMIS DAC range ($\pm 528$ nA) and at each bias compute the floating-potential operating point and the local slope $R_s = (dI/dV)^{-1}$. The minimum of $R_s$ should land near $I_{BIAS}\approx -\tfrac{1}{2}I_{ph,sat}$ (the sign convention follows the paper: a *negative* applied current means electron injection, which drives the sphere positive and thus reduces $R_s$).

$I_{BIAS}$를 쒸프하며 각 동작점에서 $R_s = (dI/dV)^{-1}$를 계산 — 최소값이 $I_{ph,sat}/2$ 근처에서 나타나야 한다.

In [ ]:
def sheath_resistance_at_bias(
    I_bias_A: float,
    n_cm3: float,
    Te_eV: float,
    Ti_eV: float,
    A_m2: float = A_SPHERE,
    A_cross_m2: float = A_SPHERE_CROSS,
    dV_eps: float = 1e-3,
) -> Tuple[float, float]:
    """Sheath resistance R_s = (dI/dV)^-1 at the biased operating point.

    Args:
        I_bias_A: Applied bias current in A (negative = electron injection).
        n_cm3, Te_eV, Ti_eV: Plasma parameters.
        A_m2: Total sphere area.
        A_cross_m2: Cross section.
        dV_eps: Finite-difference step (V).

    Returns:
        (R_s_ohm, V_op_V) at the biased operating point.
    """
    V_op = find_floating_potential(
        n_cm3, Te_eV, Ti_eV, A_m2, A_cross_m2, I_bias_A=I_bias_A
    )
    Vs = np.array([V_op - dV_eps, V_op + dV_eps])
    I_e, I_i, I_ph = probe_current_components(
        Vs, n_cm3, Te_eV, Ti_eV, A_m2, A_cross_m2
    )
    dI = (I_e[1] + I_i[1] + I_ph[1]) - (I_e[0] + I_i[0] + I_ph[0])
    R_s = (2.0 * dV_eps) / max(abs(dI), 1e-30)
    return R_s, V_op


I_bias_grid_nA = np.linspace(-528, 528, 121)
Rs_arr = np.zeros_like(I_bias_grid_nA)
Vop_arr = np.zeros_like(I_bias_grid_nA)
for k, ib in enumerate(I_bias_grid_nA):
    Rs_arr[k], Vop_arr[k] = sheath_resistance_at_bias(
        I_bias_A=ib * 1e-9, n_cm3=n_cps, Te_eV=Te_cps, Ti_eV=Ti_cps
    )

I_ph_sat_nA = (4.0e-9 * 1e4) * A_SPHERE_CROSS * 1e9
k_min = int(np.argmin(Rs_arr))
print(f"I_ph_sat (predicted)        : {I_ph_sat_nA:.1f} nA")
print(f"I_BIAS at min R_s           : {I_bias_grid_nA[k_min]:+.1f} nA")
print(f"Min R_s achieved             : {Rs_arr[k_min]:.2e} Ohm")
print(f"Unbiased R_s                 : {Rs_arr[np.argmin(np.abs(I_bias_grid_nA))]:.2e} Ohm")
print(f"R_s improvement factor       : {Rs_arr[np.argmin(np.abs(I_bias_grid_nA))]/Rs_arr[k_min]:.0f}x")

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(I_bias_grid_nA, Vop_arr)
ax[0].set_ylabel("$V_{op}$ at sphere [V]")
ax[0].axhline(0, color="k", ls=":")
ax[0].set_title("THEMIS-EFI bias optimisation: floating potential & sheath impedance")
ax[0].grid(alpha=0.3)
ax[1].semilogy(I_bias_grid_nA, Rs_arr)
ax[1].axvline(-I_ph_sat_nA / 2, color="r", ls="--", label="$-I_{ph,sat}/2$")
ax[1].axvline(-180, color="g", ls=":", label="THEMIS nominal $-180$ nA")
ax[1].set_xlabel("$I_{BIAS}$ [nA]  (negative = electron injection)")
ax[1].set_ylabel("Sheath resistance $R_s$ [$\\Omega$]")
ax[1].legend()
ax[1].grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: $\mathbf{E}\times\mathbf{B}$ drift from FGM + EFI / FGM + EFI 결합으로부터 $\mathbf{E}\times\mathbf{B}$ 드리프트

In the magnetosheath ideal MHD holds to high accuracy: $\mathbf{E} = -\mathbf{V}_i\times\mathbf{B}$. The bulk perpendicular plasma velocity is therefore the **E$\times$B drift**

$$\mathbf{V}_\perp = \frac{\mathbf{E}\times\mathbf{B}}{B^2}.$$

Bonnell et al. cross-calibrate EFI by regressing $\mathbf{E}_{EFI}$ against $-\mathbf{V}_{i,ESA}\times\mathbf{B}_{FGM}$ and reading the slope as the boom-shorting factor and the intercept as the (sunward) photoelectron offset. We reproduce this with a synthetic 30-min magnetosheath traversal: $\mathbf{B}\sim 30$ nT, $\mathbf{V}_i$ a slowly varying perpendicular flow, with EFI corrupted by a 1.45 shorting factor and a 2.5 mV/m sunward offset.

Magnetosheath에서는 ideal MHD가 잘 성립하므로, $-\mathbf{V}_i\times\mathbf{B}$와 EFI의 회귀분석으로 boom-shorting 계수와 photoelectron offset을 동시에 추산한다.

In [ ]:
def E_cross_B_drift(
    E_Vpm: NDArray,
    B_T: NDArray,
) -> NDArray:
    """Return the E x B / B^2 drift in m/s.

    Args:
        E_Vpm: E-field, shape (..., 3) in V/m.
        B_T:   B-field, shape (..., 3) in tesla.

    Returns:
        ExB / |B|^2 drift, shape (..., 3) in m/s.
    """
    B2 = np.sum(B_T**2, axis=-1, keepdims=True)
    return np.cross(E_Vpm, B_T) / np.where(B2 == 0, 1.0, B2)


def cross_calibrate_efi(
    E_efi_mVm: NDArray,
    minus_VxB_mVm: NDArray,
    component: int = 0,
) -> Tuple[float, float]:
    """Fit ``E_efi = slope * (-V x B) + intercept`` for one component.

    Slope inverse gives the boom-shorting factor; intercept is the per-spin
    photoelectron offset (sunward in DSL X).

    Args:
        E_efi_mVm: Measured EFI E in mV/m, shape (N, 3).
        minus_VxB_mVm: -V_i x B reference in mV/m, shape (N, 3).
        component: Which DSL axis to regress (0=X,1=Y,2=Z).

    Returns:
        (slope, intercept_mVm) -- slope is dimensionless; 1/slope = f_sh.
    """
    x = minus_VxB_mVm[:, component]
    y = E_efi_mVm[:, component]
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope), float(intercept)


# Synthesise 30 min @ 4 Hz of magnetosheath traversal
fs_cal = 4.0
t_cal = np.arange(0, 1800.0, 1.0 / fs_cal)
n_cal = t_cal.size

Bx = 5.0 * np.ones(n_cal)
By = 8.0 + 1.0 * np.sin(2 * np.pi * t_cal / 600.0)
Bz = 25.0 + 3.0 * np.sin(2 * np.pi * t_cal / 800.0)
B_T = np.column_stack([Bx, By, Bz]) * 1e-9  # nT -> T

Vx = -200_000.0 + 30_000.0 * np.sin(2 * np.pi * t_cal / 1200.0)  # m/s, anti-sunward sheath flow
Vy = 50_000.0 + 20_000.0 * np.sin(2 * np.pi * t_cal / 700.0)
Vz = 10_000.0 * np.sin(2 * np.pi * t_cal / 500.0)
V_mps = np.column_stack([Vx, Vy, Vz])

# Reference: ideal-MHD E = -V x B, in V/m
E_ideal = -np.cross(V_mps, B_T)

# EFI "measured" = ideal / shorting + offset + noise
f_sh = 1.45
offset_X_mVm = 2.5
noise_mVm = 0.4
E_efi = E_ideal / f_sh
E_efi[:, 0] += offset_X_mVm * 1e-3  # sunward (DSL X) offset
E_efi += noise_mVm * 1e-3 * rng.standard_normal(E_efi.shape)

# Convert to mV/m for plotting/fitting
E_efi_mVm = E_efi * 1e3
E_ideal_mVm = E_ideal * 1e3

slope_X, intercept_X = cross_calibrate_efi(E_efi_mVm, E_ideal_mVm, component=0)
slope_Y, intercept_Y = cross_calibrate_efi(E_efi_mVm, E_ideal_mVm, component=1)
print(f"DSL X regression: slope = {slope_X:.4f}, 1/slope = {1/slope_X:.3f}, intercept = {intercept_X:+.3f} mV/m")
print(f"DSL Y regression: slope = {slope_Y:.4f}, 1/slope = {1/slope_Y:.3f}, intercept = {intercept_Y:+.3f} mV/m")
print(f"Injected: f_sh = {f_sh:.3f}, sunward offset = {offset_X_mVm:.2f} mV/m")

# Recover E x B drift from corrected EFI and compare to V_perp
E_corrected = (E_efi_mVm * 1e-3) * f_sh
E_corrected[:, 0] -= intercept_X * 1e-3
V_drift_efi = E_cross_B_drift(E_corrected, B_T)

B_hat = B_T / np.linalg.norm(B_T, axis=-1, keepdims=True)
V_perp_true = V_mps - np.sum(V_mps * B_hat, axis=-1, keepdims=True) * B_hat

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].scatter(E_ideal_mVm[:, 0], E_efi_mVm[:, 0], s=2, alpha=0.3)
xline = np.linspace(E_ideal_mVm[:, 0].min(), E_ideal_mVm[:, 0].max(), 10)
axes[0, 0].plot(xline, slope_X * xline + intercept_X, "r-", lw=2,
                label=f"slope={slope_X:.3f} ({1/slope_X:.2f} f_sh)")
axes[0, 0].plot(xline, xline, "k--", lw=1, label="y=x")
axes[0, 0].set_xlabel("$-V_i\\times B$ X [mV/m]")
axes[0, 0].set_ylabel("$E_{EFI}$ X [mV/m]")
axes[0, 0].set_title("EFI vs $-V\\times B$, DSL X")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].scatter(E_ideal_mVm[:, 1], E_efi_mVm[:, 1], s=2, alpha=0.3, color="tab:orange")
axes[0, 1].plot(xline, slope_Y * xline + intercept_Y, "r-", lw=2,
                label=f"slope={slope_Y:.3f}")
axes[0, 1].plot(xline, xline, "k--", lw=1, label="y=x")
axes[0, 1].set_xlabel("$-V_i\\times B$ Y [mV/m]")
axes[0, 1].set_ylabel("$E_{EFI}$ Y [mV/m]")
axes[0, 1].set_title("EFI vs $-V\\times B$, DSL Y")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(t_cal / 60, V_perp_true[:, 0] / 1e3, label="$V_{\\perp,X}$ true")
axes[1, 0].plot(t_cal / 60, V_drift_efi[:, 0] / 1e3, ls="--", label="$E\\times B/B^2$ from EFI")
axes[1, 0].set_xlabel("Time [min]")
axes[1, 0].set_ylabel("$V_X$ [km/s]")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)
axes[1, 0].set_title("Perpendicular drift recovery")

resid = (V_drift_efi - V_perp_true) / 1e3
axes[1, 1].plot(t_cal / 60, resid[:, 0], label="$\\Delta V_X$")
axes[1, 1].plot(t_cal / 60, resid[:, 1], label="$\\Delta V_Y$")
axes[1, 1].plot(t_cal / 60, resid[:, 2], label="$\\Delta V_Z$")
axes[1, 1].set_xlabel("Time [min]")
axes[1, 1].set_ylabel("residual [km/s]")
axes[1, 1].set_title("E$\\times$B residual after calibration")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Substorm-onset E-field signature in plasma-sheet inflow / Substorm onset 시 plasma-sheet inflow E-field 시그니쳐

THEMIS–EFI's flagship requirement is to measure 1 mV/m / 10% E at 8–10 R$_E$ in the plasma sheet to time substorm onset. At onset, near-Earth reconnection drives a fast earthward bulk flow in the plasma sheet, which corresponds to an enhancement of the **dawn-dusk** convection electric field $E_y$. We synthesise an inflow event:

- Quiet pre-onset: $E_y = 0.3$ mV/m, slowly varying.
- Onset at $t_0=300$ s: $E_y$ ramps to $\sim 4$ mV/m over 60 s (corresponding to 800 km/s flow in 5 nT $B_z$ via $V = E/B$), then decays.
- Plasma sheet $B_z\sim 5$ nT slowly grows during dipolarisation.

The earthward $\mathbf{E}\times\mathbf{B}/B^2$ drift recovered should match the published 200–1000 km/s bursty bulk flows.

Substorm onset 시 근지구 reconnection이 구동하는 fast earthward flow는 plasma sheet의 dawn-dusk 전기장 $E_y$의 강화로 나타난다. THEMIS–EFI의 핵심 요구사항 (8–10 R$_E$에서 1 mV/m 정확도)을 합성 onset 이벤트로 검증한다.

In [ ]:
def synthesise_substorm_inflow(
    t_s: NDArray,
    onset_s: float = 300.0,
    rise_s: float = 60.0,
    decay_s: float = 240.0,
    Ey_quiet_mVm: float = 0.3,
    Ey_peak_mVm: float = 4.0,
    Bz_quiet_nT: float = 5.0,
    Bz_dipol_nT: float = 25.0,
    seed: int = 11,
) -> Tuple[NDArray, NDArray]:
    """Generate a synthetic plasma-sheet substorm inflow event.

    Constructs Ey(t) and Bz(t) at 8-10 R_E with: a quiet baseline; an
    asymmetric ramp/decay enhancement of dawn-dusk E starting at ``onset_s``;
    a sigmoidal dipolarisation in Bz with a small lag relative to E onset.

    Args:
        t_s: Time array (s).
        onset_s: Substorm onset time (s).
        rise_s: Ey rise time (s).
        decay_s: Ey decay e-folding time (s).
        Ey_quiet_mVm: Pre-onset baseline Ey (mV/m).
        Ey_peak_mVm: Peak Ey at onset (mV/m).
        Bz_quiet_nT: Pre-onset plasma sheet Bz (nT).
        Bz_dipol_nT: Post-dipolarisation Bz (nT).
        seed: RNG seed.

    Returns:
        (Ey_mVm, Bz_nT) arrays, same shape as t_s.
    """
    local_rng = np.random.default_rng(seed)
    Ey = np.full_like(t_s, Ey_quiet_mVm)
    after = t_s >= onset_s
    tau = t_s[after] - onset_s
    enhancement = np.where(
        tau < rise_s,
        (Ey_peak_mVm - Ey_quiet_mVm) * (tau / rise_s),
        (Ey_peak_mVm - Ey_quiet_mVm) * np.exp(-(tau - rise_s) / decay_s),
    )
    Ey[after] = Ey_quiet_mVm + enhancement

    Bz = Bz_quiet_nT + (Bz_dipol_nT - Bz_quiet_nT) * 0.5 * (
        1.0 + np.tanh((t_s - (onset_s + 30.0)) / 80.0)
    )

    Ey += 0.15 * local_rng.standard_normal(t_s.shape)
    Bz += 0.4 * local_rng.standard_normal(t_s.shape)
    return Ey, Bz


fs_ss = 4.0
t_ss = np.arange(0, 900.0, 1.0 / fs_ss)
Ey_mVm, Bz_nT = synthesise_substorm_inflow(t_ss)

# Earthward E x B drift (for E in -y and B in +z, V_x = -E_y / B_z, anti-sunward = earthward in tail)
Vx_kms = -(Ey_mVm * 1e-3) / (Bz_nT * 1e-9) / 1e3

# Detect onset by 1-mV/m threshold on a 30-s running mean Ey
win_n = int(30 * fs_ss)
kernel = np.ones(win_n) / win_n
Ey_smooth = np.convolve(Ey_mVm, kernel, mode="same")
onset_mask = Ey_smooth >= 1.0
onset_idx = int(np.argmax(onset_mask)) if onset_mask.any() else -1
onset_time_s = t_ss[onset_idx] if onset_idx >= 0 else None

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].plot(t_ss, Ey_mVm, lw=0.6, label="$E_y$ raw")
axes[0].plot(t_ss, Ey_smooth, lw=2, label="$E_y$ 30-s mean")
axes[0].axhline(1.0, color="k", ls=":", label="1 mV/m threshold")
if onset_time_s is not None:
    axes[0].axvline(onset_time_s, color="r", ls="--", label=f"detected onset = {onset_time_s:.0f} s")
axes[0].set_ylabel("$E_y$ [mV/m]")
axes[0].set_title("THEMIS-EFI substorm-onset E-field signature in plasma sheet (8–10 R$_E$)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(t_ss, Bz_nT, color="tab:purple")
axes[1].set_ylabel("$B_z$ [nT]")
axes[1].grid(alpha=0.3)
axes[1].set_title("Tail dipolarisation (FGM)")

axes[2].plot(t_ss, Vx_kms, color="tab:green")
axes[2].axhline(-200, color="k", ls=":", label="BBF threshold (-200 km/s)")
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("$V_X$ earthward = $-E_y/B_z$ [km/s]")
axes[2].legend()
axes[2].grid(alpha=0.3)
axes[2].set_title("Recovered earthward $\\mathbf{E}\\times\\mathbf{B}/B^2$ flow")
plt.tight_layout()
plt.show()

if onset_time_s is not None:
    timing_err = onset_time_s - 300.0
    print(f"True onset       : 300.0 s")
    print(f"Detected onset   : {onset_time_s:.1f} s")
    print(f"Timing error     : {timing_err:+.1f} s")
    print(f"Peak earthward V : {Vx_kms.min():.0f} km/s (BBF range 200-1000 km/s)")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Double-probe E formula / 더블 프로브 E 식 | $E = -(V_n-V_m)/L$, three orthogonal pairs (49.6 / 40.4 / 6.93 m) | MMS SDP+ADP (60 m / 14.6 m); Solar Orbiter RPW; Parker Solar Probe FIELDS |
| Sheath / floating-potential model / Sheath 모델 | OML probe currents + photoemission saturated 4 nA/cm$^2$, bias near $-I_{ph,sat}/2$ | Same OML framework used in MMS, MUSCAT, Cluster EFW spacecraft-charging models |
| Boom-shorting calibration / Boom-shorting 교정 | Cross-cal vs $-\mathbf{V}_i\times\mathbf{B}$, slope = 1/$f_{sh}$, 1.3–1.6 | All double-probe E missions still apply linear cross-cal; Polar (1.2–1.4); MMS reports 1.1–1.3 |
| $\mathbf{E}\times\mathbf{B}$ drift recovery / 드리프트 재구성 | $V_\perp = E\times B/B^2$ for ideal-MHD regions (magnetosheath) | Operational SWPC / NASA HEC products, MMS multipoint | 
| Substorm-onset E signature / Substorm-onset E 시그니쳐 | 1 mV/m sensitivity at 8–10 R$_E$ to time tail-onset | THEMIS+ARTEMIS still operational (>17 yr), MMS extends to electron scales |

### Key takeaways from the implementation / 구현으로부터의 핵심 결론
1. The double-probe formula is trivial; getting 1 mV/m DC accuracy is the entire engineering achievement — spin-fit + offset + boom-shorting all enter at the mV/m level.
2. Bias-current optimisation is not a tweak: $R_s$ falls by ~100× between unbiased and $\sim -180$ nA bias, the single most important on-orbit parameter for DC E-field accuracy.
3. EFI cross-calibration against $-\mathbf{V}_i\times\mathbf{B}$ in the magnetosheath simultaneously yields the boom-shorting factor (slope) and sunward photoelectron offset (intercept) — a self-contained calibration leveraging ideal MHD as ground truth.
4. A 1-mV/m threshold on plasma-sheet $E_y$ reliably detects the substorm onset within tens of seconds of the true ramp time, vindicating the THEMIS–EFI requirement at 8–10 R$_E$.

구현 핵심 결론: (1) Double-probe 식 자체는 단순, mV/m 정확도 달성은 공학적 도전. (2) Bias 최적화는 $R_s$를 100배 감소 — DC 정확도의 핵심. (3) Magnetosheath 교차-교정으로 boom-shorting과 photoelectron offset 동시 추출. (4) 1 mV/m 임계값으로 THEMIS 요구사항 (8–10 R$_E$ substorm onset 검출)이 몇십 초 내에 충족됨.